# LEXICON-X II | Notebook de trabajo
By Erasmo Gomez Montoya
Sube y extrae el ZIP; usa CPU para baseline.

**Versión base vectorial (vdb).** Se comparan dos sistemas que usan el mismo módulo de respuesta (*reranking por reglas*) y difieren solo en el recuperador:
- **Baseline:** TF-IDF (1–2 gramas) sobre todo el corpus.
- **Alternativa:** Qdrant (en memoria) + `intfloat/multilingual-e5-small` + filtro de payload por sitio.

Es la misma lógica que `database/index_corpus.py` y `database/query_rag.ipynb`.

In [1]:
# %pip install -q qdrant-client sentence-transformers   # descomentar en Colab
# En Colab: subir y extraer el ZIP de estudiante en /content; cambiar directorio si corresponde.
# Los archivos CSV deben estar junto al notebook.
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from pathlib import Path
D=pd.read_csv('corpus_documentos.csv'); V=pd.read_csv('validation_qa.csv',dtype={'answer':str}).fillna('')
X=D.text.tolist(); vectorizer=TfidfVectorizer(ngram_range=(1,2)); M=vectorizer.fit_transform(X)
def lexical(q,k=5):
    scores=linear_kernel(vectorizer.transform([q]),M).ravel()
    return np.argsort(-scores,kind='stable')[:k]
def evaluate(pred,gold):
    assert pred.qid.is_unique and set(pred.qid)==set(gold.qid), 'IDs incompletos o duplicados'
    z=gold.merge(pred,on='qid',suffixes=('_gold','_pred'))
    exact=(z.answer_gold.astype(str).str.strip().str.lower()==z.answer_pred.astype(str).str.strip().str.lower())
    def hit(r):
        g=set(str(r.evidence_ids).split('|'))-{'','nan'}; p=set(str(r.citations).split('|'))-{'','nan'}
        return bool(g&p) if g else not p
    h=z.apply(hit,axis=1)
    return {'EM':float(exact.mean()),'evidence_hit':float(h.mean()),'joint':float((exact&h).mean()),'by_kind':exact.groupby(z.kind).mean().to_dict()}

import json, time
SEED = 42; np.random.seed(SEED)


In [2]:
display(D.sample(8,random_state=42)); display(V.head()); print(D.groupby(["language","source"]).size())

,doc_id,site,topic,language,year,source,reliability,text
1116,D186-0,ER-186,energía,es,2048,oficial,0.95,El registro oficial de ER-186 confirma 28 celd...
1368,D228-0,ER-228,archivo,es,2048,oficial,0.95,El registro oficial de ER-228 confirma 42 tabl...
422,D070-2,ER-070,agricultura,es,2047,oficial,0.90,El registro anterior de ER-070 indica 100 semi...
413,D068-5,ER-068,archivo,es,2048,archivo,0.70,El archivo de ER-068 registra actividades de a...
451,D075-1,ER-075,agricultura,er,2048,oficial,0.95,ER-075 sava 38 ena.
861,D143-3,ER-143,archivo,er,2047,oficial,0.90,ER-143 kiri 87 ena.
1063,D177-1,ER-177,transporte,er,2048,oficial,0.95,ER-177 nora 42 ena.
741,D123-3,ER-123,archivo,er,2047,oficial,0.90,ER-123 kiri 84 ena.


,qid,question,answer,evidence_ids,kind
0,Q148-3,¿Quién dirige ER-148?,SIN_EVIDENCIA,,sin_evidencia
1,Q128-3,¿Quién dirige ER-128?,SIN_EVIDENCIA,,sin_evidencia
2,Q132-2,¿Qué cantidad de vehículos registraba ER-132 e...,95,D132-2|D132-3,historico
3,Q164-2,¿Qué cantidad de depósitos registraba ER-164 e...,38,D164-2|D164-3,historico
4,Q157-0,¿Cuántas vehículos hay disponibles actualmente...,25,D157-0|D157-1,actual


language  source 
er        oficial    500
es        archivo    250
          oficial    500
          rumor      250
dtype: int64


## Hipótesis
Detalle completo en `HIPOTESIS.md`.

- **Variable independiente:** el recuperador. Puede ser TF-IDF léxico, o denso (e5) en Qdrant con filtro de sitio. El módulo de respuesta es el mismo en ambos sistemas.
- **Métricas:** EM, `evidence_hit`, `joint`, EM por tipo (`actual`, `historico`, `sin_evidencia`) y latencia media y p95 por pregunta.

**H1 (estadística).** TF-IDF con bigramas captura el identificador exacto `ER-###` (el bigrama `er 000`), que es un token raro con IDF alto.
- *Predicción:* `joint` ≥ 0.95 en validación sin filtro de metadatos, con una latencia menor que la alternativa.
- *Se refuta si* `joint` < 0.95.

**H2 (base vectorial).** La recuperación densa con filtro de sitio en Qdrant iguala a H1 (±2 puntos de `joint`) y es más robusta a paráfrasis, a cambio de una latencia ≥ 2× mayor.
- *Predicción adicional (ablación):* sin el filtro de sitio, la alternativa densa empeora, porque los embeddings confunden `ER-000` con `ER-050`.
- *Se refuta si* su `joint` queda más de 2 puntos por debajo de H1, o si quitar el filtro no la degrada.

**Criterio de éxito global:** `joint` ≥ 0.95 en validación, sin usar las respuestas de validación como contexto ni reglas por ID.

In [3]:
print(D.iloc[lexical(V.question.iloc[0])][["doc_id","text"]])

     doc_id                                               text
889  D148-1                                ER-148 kiri 49 ena.
891  D148-3                                ER-148 kiri 56 ena.
893  D148-5  El archivo de ER-148 registra actividades de a...
890  D148-2  El registro anterior de ER-148 indica 56 tabli...
888  D148-0  El registro oficial de ER-148 confirma 49 tabl...


## Baseline y alternativa
Implementa predict y run. La salida debe tener qid, answer, citations (IDs separados por |). Usa SIN_EVIDENCIA sin citas cuando no exista respuesta. Compara ambos enfoques con evaluate.

In [4]:
# ---- Módulo común: parseo de la pregunta + reranking por reglas ----
lexicon = dict(pd.read_csv('lexico.csv').values)                 # er -> es
ITEMS = {'semillas', 'celdas', 'vehículos', 'tablillas', 'depósitos'}
CURRENT_YEAR = 2048
QUANTITY_CUES = re.compile(r'cu[aá]nt|cantidad|inventario|n[uú]mero|registraba|disponibles', re.I)
DOCS = D.to_dict(orient='records')

def translate(text):
    return ' '.join(lexicon.get(t.strip('.,').lower(), t) for t in text.split())

def parse_query(q):
    site = m.group(0).upper() if (m := re.search(r'ER-\d{3}', q, re.I)) else None
    year = int(m.group(0)) if (m := re.search(r'\b(20\d{2})\b', q)) else None
    words = {lexicon.get(w, w) for w in re.findall(r'\w+', q.lower())}
    item = next(iter(words & ITEMS), None)
    return dict(site=site, year=year, item=item, asks_quantity=bool(QUANTITY_CUES.search(q)))

def number_in(text):
    m = re.search(r'\b(\d+)\b', re.sub(r'ER-\d+', '', text))
    return m.group(1) if m else None

def rerank(pq, hits):
    """hits: lista de (indice_doc, score) ordenada por el recuperador.
    Devuelve (answer, citations). El score solo desempata; deciden los metadatos."""
    if not pq['asks_quantity']:
        return 'SIN_EVIDENCIA', []
    target_year = pq['year'] or CURRENT_YEAR
    sites = {pq['site']} if pq['site'] else {DOCS[i]['site'] for i, _ in hits}
    cands = []
    for i, s in hits:
        d = DOCS[i]
        text_es = translate(d['text']) if d['language'] == 'er' else d['text']
        if (d['source'] == 'oficial' and d['year'] == target_year and d['site'] in sites
                and number_in(d['text']) is not None
                and (pq['item'] is None or pq['item'] in text_es)):
            cands.append((d, s))
    if not cands:
        return 'SIN_EVIDENCIA', []
    best, _ = max(cands, key=lambda c: (c[0]['language'] == 'es', c[0]['reliability'], c[1]))
    value = number_in(best['text'])
    cites = [d['doc_id'] for d, _ in cands if d['site'] == best['site'] and number_in(d['text']) == value]
    cites.sort(key=lambda x: x != best['doc_id'])          # primero el documento elegido, luego su gemelo er
    return value, cites

# ---- Baseline: TF-IDF (usa `lexical` de la celda de carga) ----
def tfidf_hits(q, k=10):
    scores = linear_kernel(vectorizer.transform([q]), M).ravel()
    idx = np.argsort(-scores, kind='stable')[:k]
    return [(int(i), float(scores[i])) for i in idx]

def predict(question, retriever=tfidf_hits):
    return rerank(parse_query(question), retriever(question))

def run(questions, retriever):
    rows, lat = [], []
    for qid, q in zip(questions.qid, questions.question):
        t0 = time.perf_counter(); ans, cites = predict(q, retriever); lat.append((time.perf_counter() - t0) * 1000)
        rows.append({'qid': qid, 'answer': ans, 'citations': '|'.join(cites)})
    return pd.DataFrame(rows), np.array(lat)

print(predict(V.question.iloc[0]), '| gold:', V.answer.iloc[0], V.evidence_ids.iloc[0])

('SIN_EVIDENCIA', []) | gold: SIN_EVIDENCIA 


### Alternativa: base vectorial (Qdrant + multilingual-e5-small)
Se indexan los 1 500 documentos en un Qdrant **en memoria**: no hace falta `database/qdrant_data` y se evita el bloqueo de carpeta en Colab. Cada punto guarda el vector (384 dimensiones, coseno) y los metadatos como payload. E5 requiere los prefijos `passage:` y `query:`.

In [5]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

E5_NAME = 'intfloat/multilingual-e5-small'
e5 = SentenceTransformer(E5_NAME)
qdrant = QdrantClient(':memory:')
qdrant.create_collection('corpus', vectors_config=models.VectorParams(
    size=e5.get_embedding_dimension(), distance=models.Distance.COSINE))
vecs = e5.encode(['passage: ' + t for t in X], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
qdrant.upload_points('corpus', [models.PointStruct(id=i, vector=v.tolist(), payload=DOCS[i]) for i, v in enumerate(vecs)])

def vdb_hits(q, k=10, use_site_filter=True):
    site = parse_query(q)['site'] if use_site_filter else None
    flt = models.Filter(must=[models.FieldCondition(key='site', match=models.MatchValue(value=site))]) if site else None
    qv = e5.encode('query: ' + q, normalize_embeddings=True).tolist()
    return [(p.id, p.score) for p in qdrant.query_points('corpus', query=qv, limit=k, query_filter=flt).points]

vdb_nofilter_hits = lambda q, k=10: vdb_hits(q, k, use_site_filter=False)
print(qdrant.count('corpus').count, 'puntos |', predict(V.question.iloc[0], vdb_hits))

ModuleNotFoundError: No module named 'qdrant_client'

### Comparación en validación
Mismas 150 preguntas para los tres sistemas. Las respuestas de validación se usan **solo** en `evaluate`. Se guardan las predicciones (`qid, answer, citations`) y la tabla de resultados en el formato de `resultados_plantilla.csv`.

In [ ]:
SYSTEMS = {
    'baseline_tfidf':       (tfidf_hits,        'TF-IDF 1-2 gramas + reranking por reglas'),
    'vdb_e5_qdrant':        (vdb_hits,          'Qdrant + e5-small + filtro de sitio + reranking por reglas'),
    'ablacion_vdb_sin_filtro': (vdb_nofilter_hits, 'Igual que vdb pero sin filtro de payload por sitio'),
}
results, by_kind, preds = [], {}, {}
for name, (retriever, obs) in SYSTEMS.items():
    pred, lat = run(V, retriever)
    preds[name] = pred
    pred.to_csv(f'predicciones_validacion_{name}.csv', index=False)
    m = evaluate(pred, V)
    by_kind[name] = m['by_kind']
    results.append({'modelo': name, 'split': 'validation', 'EM': m['EM'], 'evidence_hit': m['evidence_hit'],
                    'joint': m['joint'], 'latencia_ms': round(lat.mean(), 2),
                    'latencia_p95_ms': round(np.percentile(lat, 95), 2), 'observaciones': obs})
R = pd.DataFrame(results)
R.drop(columns='latencia_p95_ms').to_csv('resultados_validacion.csv', index=False)
display(R); display(pd.DataFrame(by_kind).rename_axis('EM por tipo'))

,modelo,split,EM,evidence_hit,joint,latencia_ms,latencia_p95_ms,observaciones
0,baseline_tfidf,validation,1.000000,1.000000,1.000000,0.47,0.51,TF-IDF 1-2 gramas + reranking por reglas
1,vdb_e5_qdrant,validation,1.000000,1.000000,1.000000,22.26,32.77,Qdrant + e5-small + filtro de sitio + rerankin...
2,ablacion_vdb_sin_filtro,validation,0.933333,0.933333,0.933333,10.91,15.33,Igual que vdb pero sin filtro de payload por s...


,baseline_tfidf,vdb_e5_qdrant,ablacion_vdb_sin_filtro
EM por tipo,,,
actual,1.0,1.0,0.878378
historico,1.0,1.0,0.975610
sin_evidencia,1.0,1.0,1.000000


### Errores y casos difíciles
Se listan los errores de validación de cada sistema y se prueba un conjunto de estrés **fuera de las plantillas** de pregunta, escrito a mano: no está en train ni en validación. Para cada caso se anota la respuesta esperada según las reglas del enunciado.

In [ ]:
for name, pred in preds.items():
    z = V.merge(pred, on='qid', suffixes=('_gold', '_pred'))
    err = z[(z.answer_gold.astype(str) != z.answer_pred.astype(str)) |
            ~z.apply(lambda r: (bool(set(r.evidence_ids.split('|')) & set(str(r.citations).split('|')))
                                if r.evidence_ids else not r.citations), axis=1)]
    print(f'{name}: {len(err)} errores en validación')
    if len(err): display(err[['qid', 'question', 'kind', 'answer_gold', 'answer_pred', 'evidence_ids', 'citations']].head(10))

STRESS = [  # (pregunta, respuesta esperada, tipo de caso)
    ('¿Cuántas semillas hay disponibles actualmente en ER-000?', '93', 'contradicción 2048 vs 2047 vs rumor'),
    ('¿Qué cantidad de semillas registraba ER-000 en 2045?', 'SIN_EVIDENCIA', 'año sin registro'),
    ('¿Quién dirige ER-000?', 'SIN_EVIDENCIA', 'ausencia de evidencia'),
    ('¿Cuántas tavi hay en ER-001?', '26', 'objeto en Eráviri'),
    ('¿Cuántos habitantes hay en ER-000?', 'SIN_EVIDENCIA', 'objeto inexistente'),
    ('¿Qué cantidad de semillas había en ER-000 el año pasado?', '100', 'fecha relativa'),
    ('¿Cuántas semillas hay en el sitio 0?', '93', 'sitio sin formato ER-###'),
    ('¿Cuántas semillas hay disponibles?', 'SIN_EVIDENCIA', 'sin sitio'),
    ('¿Cuál es el stock de semillas de ER-000 hoy?', '93', 'paráfrasis'),
]
S = pd.DataFrame(STRESS, columns=['question', 'esperado', 'caso'])
for name, (retriever, _) in SYSTEMS.items():
    S[name] = [predict(q, retriever)[0] for q in S.question]
display(S)
print((S[list(SYSTEMS)].values == S[['esperado']].values).mean(axis=0).round(2), '← aciertos en estrés por sistema')

baseline_tfidf: 0 errores en validación
vdb_e5_qdrant: 0 errores en validación
ablacion_vdb_sin_filtro: 10 errores en validación


,qid,question,kind,answer_gold,answer_pred,evidence_ids,citations
42,Q134-2,¿Qué cantidad de depósitos registraba ER-134 e...,historico,68,SIN_EVIDENCIA,D134-2|D134-3,
51,Q168-1,"Según el registro vigente, ¿cuál es el inventa...",actual,79,SIN_EVIDENCIA,D168-0|D168-1,
55,Q143-1,"Según el registro vigente, ¿cuál es el inventa...",actual,80,SIN_EVIDENCIA,D143-0|D143-1,
86,Q165-1,"Según el registro vigente, ¿cuál es el inventa...",actual,59,SIN_EVIDENCIA,D165-0|D165-1,
89,Q140-1,"Según el registro vigente, ¿cuál es el inventa...",actual,82,SIN_EVIDENCIA,D140-0|D140-1,
91,Q160-1,"Según el registro vigente, ¿cuál es el inventa...",actual,93,SIN_EVIDENCIA,D160-0|D160-1,
92,Q150-1,"Según el registro vigente, ¿cuál es el inventa...",actual,32,SIN_EVIDENCIA,D150-0|D150-1,
93,Q133-1,"Según el registro vigente, ¿cuál es el inventa...",actual,20,SIN_EVIDENCIA,D133-0|D133-1,
124,Q131-1,"Según el registro vigente, ¿cuál es el inventa...",actual,66,SIN_EVIDENCIA,D131-0|D131-1,
142,Q171-1,"Según el registro vigente, ¿cuál es el inventa...",actual,53,SIN_EVIDENCIA,D171-0|D171-1,


,question,esperado,caso,baseline_tfidf,vdb_e5_qdrant,ablacion_vdb_sin_filtro
0,¿Cuántas semillas hay disponibles actualmente ...,93,contradicción 2048 vs 2047 vs rumor,93,93,93
1,¿Qué cantidad de semillas registraba ER-000 en...,SIN_EVIDENCIA,año sin registro,SIN_EVIDENCIA,SIN_EVIDENCIA,SIN_EVIDENCIA
2,¿Quién dirige ER-000?,SIN_EVIDENCIA,ausencia de evidencia,SIN_EVIDENCIA,SIN_EVIDENCIA,SIN_EVIDENCIA
3,¿Cuántas tavi hay en ER-001?,26,objeto en Eráviri,26,26,SIN_EVIDENCIA
4,¿Cuántos habitantes hay en ER-000?,SIN_EVIDENCIA,objeto inexistente,93,93,93
5,¿Qué cantidad de semillas había en ER-000 el a...,100,fecha relativa,93,93,93
6,¿Cuántas semillas hay en el sitio 0?,93,sitio sin formato ER-###,39,23,23
7,¿Cuántas semillas hay disponibles?,SIN_EVIDENCIA,sin sitio,39,41,41
8,¿Cuál es el stock de semillas de ER-000 hoy?,93,paráfrasis,SIN_EVIDENCIA,SIN_EVIDENCIA,SIN_EVIDENCIA


[0.44 0.44 0.33] ← aciertos en estrés por sistema


## Recursos opcionales: embeddings y generación
Descomenta instalación si es necesaria. No son obligatorios; mide latencia y calidad.

In [ ]:
# Opcional: ejecutar en Colab con Internet y GPU si está disponible.
# %pip install -q sentence-transformers transformers sentencepiece
from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
E=encoder.encode(X,normalize_embeddings=True,show_progress_bar=True)
def dense(q,k=5):
    v=encoder.encode([q],normalize_embeddings=True)[0]
    return np.argsort(-(E@v))[:k]



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18485.70it/s]


Batches:   0%|          | 0/47 [00:00<?, ?it/s]


Batches:   2%|▏         | 1/47 [00:00<00:09,  5.08it/s]


Batches:  11%|█         | 5/47 [00:00<00:02, 18.64it/s]


Batches:  21%|██▏       | 10/47 [00:00<00:01, 28.00it/s]

... (4 outputs left out)


Batches:  72%|███████▏  | 34/47 [00:01<00:00, 39.99it/s]


Batches:  85%|████████▌ | 40/47 [00:01<00:00, 43.82it/s]


Batches:  96%|█████████▌| 45/47 [00:01<00:00, 44.87it/s]


Batches: 100%|██████████| 47/47 [00:01<00:00, 32.93it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
name='google/flan-t5-small'; tok=AutoTokenizer.from_pretrained(name); model=AutoModelForSeq2SeqLM.from_pretrained(name)
device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device).eval()
def generate(q,idx):
    context='\n'.join(f'{D.iloc[i].doc_id}: {D.iloc[i].text}' for i in idx)
    prompt=f'Responde solo con el número solicitado o SIN_EVIDENCIA. Pregunta: {q} Evidencia: {context}'
    inputs=tok(prompt,return_tensors='pt',truncation=True,max_length=512).to(device)
    with torch.no_grad(): out=model.generate(**inputs,max_new_tokens=20)
    return tok.decode(out[0],skip_special_tokens=True)
print(generate(V.question.iloc[0],dense(V.question.iloc[0],5)))



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 8919.76it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


D157-3: ER-157 nora 32 ena.
